# RAG-IDEArq — Indexación GeoJSON en Weaviate

Indexa los yacimientos del GeoJSON `c14_v2.geojson` en las colecciones existentes de Weaviate.

**Requisitos:**
- Las colecciones de PDFs ya deben existir (MiniLM, GTE, E5)
- Weaviate debe estar corriendo en localhost:8080

**Características:**
- E5 con clase custom (prefijos passage/query)
- GTE con trust_remote_code=True
- Batch size 5 para evitar OOM
- Retry mechanism si falla un batch
- Schema completo (22 propiedades)

In [1]:
# Cell 1: Imports
import os
import sys
import json
import gc
import time
from pathlib import Path
from typing import List, Dict, Any

from dotenv import load_dotenv

import torch
import weaviate
from langchain_weaviate import WeaviateVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer

print('Imports OK')

Imports OK


In [2]:
# Cell 2: Setup paths y config
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

from src.config import EMBEDDINGS, collection_name, GEOJSON_DIR, INDEX_PROPERTIES_FULL

print(f"Project root: {PROJECT_ROOT}")
print(f"GeoJSON path: {GEOJSON_DIR}")
print(f"Embeddings: {list(EMBEDDINGS.keys())}")
print(f"Schema properties: {len(INDEX_PROPERTIES_FULL)}")

Project root: /home/raglinux/RAG
GeoJSON path: /home/raglinux/RAG/data/ingesta/geojson/c14_v2.geojson
Embeddings: ['all-MiniLM-L6-v2', 'gte-multilingual-base', 'e5-large-instruct']
Schema properties: 23


In [3]:
# Cell 3: E5 Custom Embedding
class E5InstructEmbeddings(Embeddings):
    """Custom embedding class for E5 Instruct models.
    
    E5 requires special prefixes:
    - "passage: " for documents during indexing
    - "query: " for queries during retrieval
    """
    
    def __init__(self, model_name="intfloat/multilingual-e5-large-instruct", device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.model = SentenceTransformer(model_name, device=self.device)
    
    def embed_documents(self, texts):
        prefixed = [f"passage: {t}" for t in texts]
        return self.model.encode(prefixed, device=self.device).tolist()
    
    def embed_query(self, text):
        prefixed = f"query: {text}"
        return self.model.encode([prefixed], device=self.device)[0].tolist()

print("E5InstructEmbeddings class defined")

E5InstructEmbeddings class defined


In [4]:
# Cell 4: Helper functions
def safe_empty_cache():
    """Clean GPU memory to prevent OOM errors."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_geojson(geojson_path: str) -> List[Document]:
    """Load c14_v2.geojson as Documents."""
    path = Path(geojson_path)
    if path.is_dir():
        geo_path = path / "c14_v2.geojson"
    else:
        geo_path = path
    
    if not geo_path.exists():
        print(f"ERROR: {geo_path} not found")
        return []

    print(f"Loading {geo_path.name}...")
    all_docs = []
    try:
        with open(geo_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        features = data.get('features', [])
        for idx, feat in enumerate(features):
            geom = feat.get('geometry') or {}
            if geom.get('type') != 'Point':
                continue
            coords = geom.get('coordinates') or []
            if len(coords) < 2:
                continue

            props = feat.get('properties') or {}
            lon, lat = float(coords[0]), float(coords[1])

            text_parts = []
            nombre = props.get('yacimiento', props.get('name', 'Desconocido'))
            text_parts.append(f"Yacimiento: {nombre}")
            if props.get('unidad_territorial'):
                text_parts.append(f"Ubicación: {props['unidad_territorial']}")
            if props.get('tipologia_crono'):
                text_parts.append(f"Tipología: {props['tipologia_crono']}")
            if props.get('descripcion'):
                desc = str(props['descripcion'])[:500]
                text_parts.append(f"Descripción: {desc}")
            if props.get('dataciones_c_14'):
                text_parts.append(f"Dataciones C14: {props['dataciones_c_14']}")

            doc = Document(
                page_content="\n".join(text_parts),
                metadata={
                    'source': str(geo_path),
                    'filename': geo_path.name,
                    'doc_type': 'yacimiento',
                    'doc_index': idx,
                    'lat': lat,
                    'lon': lon,
                    'yacimiento_id': props.get('yacimiento_id', props.get('id')),
                    'yacimiento_nombre': nombre,
                    'unidad_territorial': props.get('unidad_territorial', ''),
                    'tipologia_crono': props.get('tipologia_crono', ''),
                    'title': nombre,
                    'language': 'es',
                    'chunking_method': 'geojson_1doc_1feature',
                }
            )
            all_docs.append(doc)
    except Exception as e:
        print(f"  Error loading {geo_path.name}: {e}")

    print(f"Loaded {len(all_docs)} yacimiento documents from {geo_path.name}")
    return all_docs

print("Functions defined")

Functions defined


In [5]:
# Cell 5: Load embedding model function
def load_embedding_model(emb_cfg: Dict[str, Any]):
    """Load the appropriate embedding model based on configuration."""
    model_name = emb_cfg["model_name"]
    model_class = emb_cfg.get("model_class", "HuggingFaceEmbeddings")
    
    # E5 requires custom class with passage/query prefixes
    if model_class == "E5InstructEmbeddings":
        try:
            emb = E5InstructEmbeddings(model_name=model_name, device="cuda")
            return emb, "cuda"
        except Exception as e:
            print(f"  [E5 CUDA failed: {e}] Falling back to CPU...")
            emb = E5InstructEmbeddings(model_name=model_name, device="cpu")
            return emb, "cpu"
    
    # Standard HuggingFaceEmbeddings
    model_kwargs = {"device": "cuda"}
    encode_kwargs = {"device": "cuda"}
    
    # GTE requires trust_remote_code=True
    if emb_cfg.get("trust_remote_code", False):
        model_kwargs["trust_remote_code"] = True
    
    try:
        emb = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs=model_kwargs,
            encode_kwargs=encode_kwargs,
        )
        return emb, "cuda"
    except Exception as e:
        print(f"  [CUDA failed: {e}] Falling back to CPU...")
        model_kwargs["device"] = "cpu"
        encode_kwargs["device"] = "cpu"
        emb = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs=model_kwargs,
            encode_kwargs=encode_kwargs,
        )
        return emb, "cpu"

print("load_embedding_model function defined")

load_embedding_model function defined


In [ ]:
# Cell 6: Index function
def index_geojson_to_collection(emb_key: str, emb_cfg: Dict[str, Any], 
                                 geojson_docs: List[Document], w_client) -> Dict[str, Any]:
    """Index GeoJSON docs into an existing collection."""
    coll_name = collection_name(emb_key)
    print(f"\n{'='*60}")
    print(f"Embedding: {emb_key} → {coll_name}")
    print(f"GeoJSON docs to add: {len(geojson_docs)}")
    print(f"{'='*60}")

    if not w_client.collections.exists(coll_name):
        print(f"  ERROR: Collection '{coll_name}' does not exist.")
        print(f"  Run the PDF indexing notebook first.")
        return {"indexed": 0, "errors": len(geojson_docs), "object_count": 0}

    # Load embedding model with appropriate class
    emb, device = load_embedding_model(emb_cfg)
    print(f"  Model loaded on {device}: {emb_cfg['model_name']}")

    # Create vector store with FULL schema attributes
    vs = WeaviateVectorStore(
        client=w_client,
        index_name=coll_name,
        text_key="content",
        embedding=emb,
        attributes=[p[0] for p in INDEX_PROPERTIES_FULL if p[0] != "content"],
    )

    # Add docs in batches with retry mechanism
    batch_size = 5
    indexed = 0
    errors = 0
    t0 = time.time()

    for i in range(0, len(geojson_docs), batch_size):
        batch = geojson_docs[i:i + batch_size]
        try:
            vs.add_documents(batch)
            indexed += len(batch)
            
            if indexed % 50 == 0:
                elapsed = time.time() - t0
                rate = indexed / elapsed if elapsed > 0 else 0
                print(f"  [{indexed}/{len(geojson_docs)}] {rate:.1f} docs/s")
                
        except Exception as e:
            errors += len(batch)
            print(f"  Error at batch {i}: {e}")
            safe_empty_cache()
            
            # Retry one by one
            print(f"  Retrying {len(batch)} docs individually...")
            for doc in batch:
                try:
                    vs.add_documents([doc])
                    indexed += 1
                    errors -= 1  # One succeeded
                except Exception as e2:
                    print(f"    Failed: {e2}")

    elapsed = time.time() - t0

    # Verify total objects in collection
    coll = w_client.collections.get(coll_name)
    agg = coll.aggregate.over_all(total_count=True)
    obj_count = agg.total_count or 0

    print(f"\n  Results:")
    print(f"    Indexed: {indexed}")
    print(f"    Errors: {errors}")
    print(f"    Time: {elapsed:.1f}s ({indexed/elapsed:.1f} docs/s)")
    print(f"    Total objects in collection: {obj_count}")
    
    return {
        "indexed": indexed,
        "errors": errors,
        "object_count": obj_count,
        "time_s": elapsed,
        "device": device,
    }

print("index_geojson_to_collection function defined")

✅ index_geojson_to_collection function defined


## Ejecución

Las siguientes celdas ejecutan la indexación paso a paso.

In [7]:
# Cell 7: Conectar a Weaviate
w_client = weaviate.connect_to_local(
    host="localhost",
    port=8080,
    grpc_port=50051,
)
print(f"Weaviate: {w_client.is_ready()}")
print(f"Existing collections:")
for name in sorted(w_client.collections.list_all().keys()):
    coll = w_client.collections.get(name)
    agg = coll.aggregate.over_all(total_count=True)
    print(f"  {name}: {agg.total_count} objetos")

Weaviate: True
Existing collections:
  BgeE5: 0 objetos
  BgeE5Chunk1000Sim05: 215662 objetos
  IdearqGTE_800_50_v2: 53758 objetos
  IdearqLinq: 9035 objetos
  IdearqMiniLM_800_50_v2: 53758 objetos
  IdearqQwen3: 9035 objetos
  Idearqe5largeinstruct_800_50_v2: 53758 objetos


/home/raglinux/env_rag/lib/python3.12/site-packages/weaviate/warnings.py:93: DeprecationWarning: Dep005: You are using weaviate-client version 4.17.0. The latest version is 4.22.0.
            Consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(


In [8]:
# Cell 8: Cargar GeoJSON
geojson_docs = load_geojson(GEOJSON_DIR)
if not geojson_docs:
    print("ERROR: No GeoJSON docs loaded.")
    w_client.close()
else:
    print(f"{len(geojson_docs)} yacimientos cargados")

Loading c14_v2.geojson...
Loaded 4478 yacimiento documents from c14_v2.geojson
4478 yacimientos cargados


In [11]:
# Cell 9: Indexar en cada colección
if geojson_docs:
    results = []
    for emb_key, emb_cfg in EMBEDDINGS.items():
        result = index_geojson_to_collection(emb_key, emb_cfg, geojson_docs, w_client)
        results.append((emb_key, result))
        safe_empty_cache()
    
    # Final summary
    print("\n" + "="*60)
    print("FINAL SUMMARY")
    print("="*60)
    for emb_key, result in results:
        coll_name = collection_name(emb_key)
        status = "OK" if result["errors"] == 0 else "⚠️  WARN"
        print(f"{status} {emb_key:30s} → {coll_name}")
        print(f"     Indexed: {result['indexed']}, Errors: {result['errors']}, "
              f"Total: {result['object_count']}, Time: {result['time_s']:.1f}s")
    
    w_client.close()
    print("\nTerminado")


Embedding: all-MiniLM-L6-v2 → IdearqMiniLM_800_50_v2
GeoJSON docs to add: 4478
  Model loaded on cuda: sentence-transformers/all-MiniLM-L6-v2
  [50/4478] 3.9 docs/s
  [100/4478] 4.2 docs/s
  [150/4478] 4.4 docs/s
  [200/4478] 4.5 docs/s
  [250/4478] 4.5 docs/s
  [300/4478] 4.6 docs/s
  [350/4478] 4.6 docs/s
  [400/4478] 4.6 docs/s
  [450/4478] 4.6 docs/s
  [500/4478] 4.6 docs/s
  [550/4478] 4.6 docs/s
  [600/4478] 4.6 docs/s
  [650/4478] 4.6 docs/s
  [700/4478] 4.6 docs/s
  [750/4478] 4.7 docs/s
  [800/4478] 4.7 docs/s
  [850/4478] 4.7 docs/s
  [900/4478] 4.7 docs/s
  [950/4478] 4.7 docs/s
  [1000/4478] 4.7 docs/s
  [1050/4478] 4.7 docs/s
  [1100/4478] 4.7 docs/s
  [1150/4478] 4.7 docs/s
  [1200/4478] 4.7 docs/s
  [1250/4478] 4.7 docs/s
  [1300/4478] 4.7 docs/s
  [1350/4478] 4.7 docs/s
  [1400/4478] 4.7 docs/s
  [1450/4478] 4.7 docs/s
  [1500/4478] 4.7 docs/s
  [1550/4478] 4.7 docs/s
  [1600/4478] 4.7 docs/s
  [1650/4478] 4.7 docs/s
  [1700/4478] 4.7 docs/s
  [1750/4478] 4.7 docs/s
  

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


  Model loaded on cuda: Alibaba-NLP/gte-multilingual-base
  [50/4478] 4.3 docs/s
  [100/4478] 4.5 docs/s
  [150/4478] 4.5 docs/s
  [200/4478] 4.5 docs/s
  [250/4478] 4.5 docs/s
  [300/4478] 4.5 docs/s
  [350/4478] 4.5 docs/s
  [400/4478] 4.5 docs/s
  [450/4478] 4.5 docs/s
  [500/4478] 4.5 docs/s
  [550/4478] 4.5 docs/s
  [600/4478] 4.5 docs/s
  [650/4478] 4.5 docs/s
  [700/4478] 4.5 docs/s
  [750/4478] 4.5 docs/s
  [800/4478] 4.5 docs/s
  [850/4478] 4.5 docs/s
  [900/4478] 4.5 docs/s
  [950/4478] 4.5 docs/s
  [1000/4478] 4.5 docs/s
  [1050/4478] 4.5 docs/s
  [1100/4478] 4.5 docs/s
  [1150/4478] 4.5 docs/s
  [1200/4478] 4.5 docs/s
  [1250/4478] 4.5 docs/s
  [1300/4478] 4.5 docs/s
  [1350/4478] 4.5 docs/s
  [1400/4478] 4.5 docs/s
  [1450/4478] 4.5 docs/s
  [1500/4478] 4.5 docs/s
  [1550/4478] 4.5 docs/s
  [1600/4478] 4.5 docs/s
  [1650/4478] 4.5 docs/s
  [1700/4478] 4.5 docs/s
  [1750/4478] 4.5 docs/s
  [1800/4478] 4.5 docs/s
  [1850/4478] 4.5 docs/s
  [1900/4478] 4.5 docs/s
  [1950/4478